## 1. Verify GPU is enabled
Checking that Google Colab is using an NVIDIA GPU for YOLO training.

In [1]:
!nvidia-smi

Sun Nov 30 04:30:28 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Import Dataset from Roboflow
Load the augmented dataset prepared in Roboflow into Google Colab for YOLOv8 training.


In [2]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="KnH9wRnXxJoDn9q61teK")
project = rf.workspace("stlab").project("arabic-sign-language-wx7v7-dzmzp")
version = project.version(1)
dataset = version.download("yolov8")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 108.6 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to arabic-sign-language-1 in yolov8:: 100%|██████████| 15094/15094 [00:02<00:00, 6789.77it/s]


## 3. Install YOLOv8

Install the `ultralytics` library, which provides YOLOv8 for training and evaluation.

In [3]:
%pip install ultralytics

from ultralytics import YOLO
from roboflow import Roboflow
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## 4. Train YOLOv8s on the Roboflow Dataset

In this step, we will train the YOLOv8 small model (`yolov8s.pt`)
on the dataset downloaded from Roboflow.  


In [4]:

DATA_YAML_PATH = "/content/arabic-sign-language-1/data.yaml"

model = YOLO("yolov8s.pt")

results = model.train(
    data=DATA_YAML_PATH,
    epochs=30,
    imgsz=640,
    batch=16,
    device=0
)


Ultralytics 8.3.233 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/arabic-sign-language-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=Tr

## 5. Saving the trained model to google drive

In [9]:
# Create folder in my Drive
!mkdir -p "/content/drive/MyDrive/yolo_project"

# Save best.pt
!cp "/content/runs/detect/train/weights/best.pt" "/content/drive/MyDrive/yolo_project/best.pt"

# Save training run
!cp -r "/content/runs/detect/train" "/content/drive/MyDrive/yolo_project/train_run"

## 6. Evaluate YOLOv8s on the Validation Set

In [10]:
from ultralytics import YOLO

WEIGHTS_PATH = "/content/runs/detect/train/weights/best.pt"
DATA_YAML_PATH = "/content/arabic-sign-language-1/data.yaml"

model = YOLO(WEIGHTS_PATH)

# Evaluate on the validation split
metrics = model.val(
    data=DATA_YAML_PATH,
    split='val',
    imgsz=640,
    batch=16
)

print("Validation Results")
print("====================")
print("Overall mAP50:     ", float(metrics.box.map50))
print("Overall mAP50-95:  ", float(metrics.box.map))

# Print per-class mAP50-95
print("\nPer-class mAP50-95:")
for class_name, score in zip(model.names.values(), metrics.box.maps):
    print(f"{class_name}: {score:.3f}")

Ultralytics 8.3.233 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 11,136,807 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1281.2±532.1 MB/s, size: 29.8 KB)
val: Scanning /content/arabic-sign-language-1/valid/labels.cache... 289 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 289/289 602.5Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 2.6it/s 7.3s
                   all        289        289      0.976      0.973      0.992       0.78
                   ain          8          8      0.992          1      0.995      0.801
                 aleff         11         11          1      0.955      0.995      0.853
                    ba         11         11          1      0.665      0.995      0.822
                   dal          3          3      0.974          1      0.995      0.587
                   dha         

## 6. Evaluate YOLOv8s on the Test Set

---



- Intersection over Union (IoU)
- Mean Average Precision (mAP) per class and overall
- Inference Speed (FPS)
- FLOPs and Model Size

In [11]:
from ultralytics import YOLO
import os

WEIGHTS_PATH = "/content/runs/detect/train/weights/best.pt"
DATA_YAML_PATH = "/content/arabic-sign-language-1/data.yaml"

model = YOLO(WEIGHTS_PATH)

metrics = model.val(
    data=DATA_YAML_PATH,
    split="test",
    imgsz=640,
    batch=16,
    conf=0.5 ,
    save = True
)

# mAP + IoU results
print("\n===== Detection Performance =====")
print("Overall mAP50-95 :", float(metrics.box.map))
print("Overall mAP50    :", float(metrics.box.map50))
print("\nPer-class mAP50 values:")
for class_name, score in zip(model.names.values(), metrics.box.maps):
    print(f"{class_name}: {score:.3f}")

# Inference speed
inf_ms = metrics.speed['inference']
fps = 1000 / inf_ms
print("\n===== Inference Speed =====")
print(f"Inference time per image: {inf_ms:.2f} ms")
print(f"FPS: {fps:.2f}")

# FLOPs + model size
print("\n===== Model Complexity =====")
model.info()  # prints Params + GFLOPs

size_mb = os.path.getsize(WEIGHTS_PATH) / (1024**2)
print(f"Model file size: {size_mb:.2f} MB")


Ultralytics 8.3.233 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 11,136,807 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 13.6±6.2 MB/s, size: 32.6 KB)
val: Scanning /content/arabic-sign-language-1/test/labels... 289 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 289/289 1.0Kit/s 0.3s
val: New cache created: /content/arabic-sign-language-1/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 3.6it/s 5.3s
                   all        289        289      0.986      0.964      0.979      0.795
                   ain          9          9      0.996          1      0.995      0.776
                 aleff         13         13          1          1      0.995      0.767
                    ba          8          8          1       0.75      0.875      0.775
                   dal         10         10          1    

## 6. Sample Detection Outputs (for Presentation)

In [12]:
import glob
from IPython.display import Image, display

# Change this if your folder is predict2, predict3, etc.
pred_folder = "/content/runs/detect/val2"

sample_imgs = glob.glob(pred_folder + "/*.jpg")[:6]   # show first 6 images

for img_path in sample_imgs:
    display(Image(filename=img_path))

Output hidden; open in https://colab.research.google.com to view.

In [14]:
from ultralytics import YOLO

WEIGHTS_PATH = "/content/runs/detect/train/weights/best.pt"
TEST_IMAGES  = "/content/arabic-sign-language-1/test/images"

model = YOLO(WEIGHTS_PATH)

results = model.predict(
    source=TEST_IMAGES,
    imgsz=640,
    conf=0.1,   # lower threshold
    save=True,
    project="runs/detect",
    name="test_lowconf"
)

print("Saved to: runs/detect/test_lowconf")


image 1/289 /content/arabic-sign-language-1/test/images/1001_46_F_bb_6_jpg.rf.18afc5f6050734f0605ca6c9ade55484.jpg: 640x640 1 ba, 16.3ms
image 2/289 /content/arabic-sign-language-1/test/images/1001_46_F_dha_2_jpg.rf.4d9f7a3316d43c2b22386d53115c86af.jpg: 640x640 1 dha, 16.3ms
image 3/289 /content/arabic-sign-language-1/test/images/1001_46_F_fa_2_jpg.rf.7738ae4c998d83c027fb7d6409073c85.jpg: 640x640 1 fa, 16.3ms
image 4/289 /content/arabic-sign-language-1/test/images/1001_46_F_gaaf_5_jpg.rf.f317f1d1cceea035a7f68f2152c8b8b0.jpg: 640x640 1 gaaf, 16.3ms
image 5/289 /content/arabic-sign-language-1/test/images/1001_46_F_ghain_0_jpg.rf.1b53e39b7163e99f6b97e4679ef5ec7d.jpg: 640x640 1 ghain, 15.0ms
image 6/289 /content/arabic-sign-language-1/test/images/1001_46_F_ghain_2_jpg.rf.ef475401f22366e46237f1993daced25.jpg: 640x640 1 ghain, 15.0ms
image 7/289 /content/arabic-sign-language-1/test/images/1001_46_F_haa_8_jpg.rf.26979e53c7b1df09c649a37d1da2bb5a.jpg: 640x640 1 haa, 15.0ms
image 8/289 /content

## 7. Real-Time Detection Using Webcam

In [ ]:
from ultralytics import YOLO
import cv2

# 1) Load your trained model (path is what you used before)
model = YOLO("bestv8.pt")

# 2) Open webcam (on Windows sometimes CAP_DSHOW helps)
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

if not cap.isOpened():
    print("❌ Could not open webcam")
else:
    print("✅ Webcam opened")

# 3) Create a resizable window
cv2.namedWindow("YOLO Webcam Detection", cv2.WINDOW_NORMAL)

while True:
    ret, frame = cap.read()
    if not ret:
        print("❌ Failed to read frame from webcam")
        break

    # Run YOLO
    results = model.predict(frame, conf=0.5, imgsz=640, verbose=False)
    annotated = results[0].plot()

    # Show frame
    cv2.imshow("YOLO Webcam Detection", annotated)

    # Press q to quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

FileNotFoundError: [Errno 2] No such file or directory: 'best.pt'